# Poisson: conditioning and accuracy versus dimension

The solution operator $K = \Delta^{-1}: f \mapsto u$ of the 2-D Poisson problem $\Delta u = f$ on $(0,1)^2$, $u = 0$ on the boundary, is linear.

* Inputs and outputs are expanded in the sine basis $\xi_{\mathbf n}(x) = 2\sin(\pi n_1 x_1)\sin(\pi n_2 x_2)$, $\mathbf n \in [35]^2$ (1225 coefficients). $\Delta$ is diagonal in this basis, so $(Kf)_{\mathbf n} = -f_{\mathbf n} / (\pi^2 |\mathbf n|^2)$.
* Input measure: independent coefficients $f_{\mathbf n} \sim \mathrm{Jacobi}(\alpha_{\mathbf n}, \alpha_{\mathbf n})$ with $\alpha_{\mathbf n} = \|\mathbf n\|_1^3$. The 1225 modes carry 95% of the input energy.
* Approximation space: linear operators acting on the first $N_{\mathrm{eff}}$ input modes (lexicographic order), i.e. the index set $\{e_1, \dots, e_{N_{\mathrm{eff}}}\}$ (`owls.linear_index_set`).

For each $N_{\mathrm{eff}}$ we draw $M$ = `owls.sample_size(N_eff)` training inputs, either from the optimal measure $\mu$ or from $\rho$ (Monte Carlo), and record the condition number of the Gram matrix and the empirical Bochner error on 500 test inputs $f \sim \rho$ (3 repetitions).

**Status:** rerun after the cleanup. The arXiv v1 version of this experiment had a bug: the Monte Carlo runs always used 500 training samples instead of $M$. See `ERRATA.md`.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters
K_MODES = 35                      # sine modes per direction (d = 35^2 = 1225)
DIMS = np.arange(100, 1300, 100)  # values of N_eff
RUNS = 3                          # repetitions per N_eff
M_TEST = 500                      # test samples from rho
SEED = 0
if QUICK:
    K_MODES, DIMS, RUNS, M_TEST = 8, np.array([10, 30]), 2, 50

n1, n2 = np.meshgrid(np.arange(1, K_MODES + 1), np.arange(1, K_MODES + 1), indexing="ij")
n1, n2 = n1.flatten(), n2.flatten()                              # modes n = (n1, n2) in lexicographic order
measures = [owls.Jacobi((a + b) ** 3) for a, b in zip(n1, n2)]  # f_n ~ Jacobi(alpha_n, alpha_n), alpha_n = ||n||_1^3
eigs = -1 / (np.pi**2 * (n1**2 + n2**2))                        # K is diagonal: (Kf)_n = eigs_n * f_n
d = len(measures)                                                # number of input (and output) coefficients

In [ ]:
if RECOMPUTE or QUICK:
    cond = {True: np.zeros((len(DIMS), RUNS)), False: np.zeros((len(DIMS), RUNS))}
    err = {True: np.zeros((len(DIMS), RUNS)), False: np.zeros((len(DIMS), RUNS))}
    for i, N in enumerate(DIMS):
        L = owls.linear_index_set(N, d)
        M = owls.sample_size(N)
        for r in range(RUNS):
            for optimal in (True, False):
                rng = np.random.default_rng([SEED, i, r, int(optimal)])
                if optimal:
                    x = owls.sample_optimal(M, L, measures, rng)
                else:
                    x = owls.sample_rho(M, measures, rng)
                model = owls.BlockWLS(L, measures).fit(x, x * eigs, optimal=optimal)
                x_test = owls.sample_rho(M_TEST, measures, rng)
                cond[optimal][i, r] = model.cond()
                err[optimal][i, r] = owls.bochner_error(model.predict(x_test), x_test * eigs)
                del model, x
        print(f"N_eff = {N:5d}, M = {M:6d}: mean cond(G) optimal {cond[True][i].mean():.2f}, "
              f"Monte Carlo {cond[False][i].mean():.2f}")

    np.save(os.path.join(DATA, "Dims.npy"), DIMS)
    np.save(os.path.join(DATA, "Condition_Ind.npy"), cond[True])
    np.save(os.path.join(DATA, "Condition_Std.npy"), cond[False])
    np.save(os.path.join(DATA, "Err_Ind.npy"), err[True])
    np.save(os.path.join(DATA, "Err_Std.npy"), err[False])

In [ ]:
dims = np.load(os.path.join(DATA, "Dims.npy"))
results = {
    "cond": (np.load(os.path.join(DATA, "Condition_Ind.npy")), np.load(os.path.join(DATA, "Condition_Std.npy"))),
    "err": (np.load(os.path.join(DATA, "Err_Ind.npy")), np.load(os.path.join(DATA, "Err_Std.npy"))),
}

fig_width, fig_height = paper_style(width=1.0, aspect=0.35, fontsize=7)
fig, ax = plt.subplots(ncols=2, figsize=(fig_width, fig_height), sharex=True, layout="constrained")
for a, key in enumerate(["cond", "err"]):
    for vals, color in zip(results[key], ["r", "k"]):
        for i, N in enumerate(dims):  # individual runs and their spread
            ax[a].plot(np.full(vals.shape[1], N), vals[i], color + "o", markersize=1)
            ax[a].plot([N, N], [vals[i].min(), vals[i].max()], color + "-.", linewidth=0.5)
    ax[a].plot(dims, results[key][0].mean(axis=1), "r^-.", markersize=4, linewidth=1, label=r"Optimal Sampling: $f^i\sim \mu$")
    ax[a].plot(dims, results[key][1].mean(axis=1), "kv--", markersize=4, linewidth=1, label=r"Standard Sampling: $f^i\sim \rho$")
    ax[a].set(yscale="log")
    ax[a].legend(loc="best")
ax[0].set(ylabel=r"$\mathrm{cond}(\mathbf{G})$", title="Stability")
ax[1].set(ylabel=r"$\sqrt{\mathbb{E}\Vert Kf^i - \tilde{K}_V f^i \Vert_{\mathcal{Y}}^2}$", title="Accuracy")
fig.supxlabel(r"$N_{\mathrm{eff}}$")
savefig(os.path.join(FIGS, "cond_err.pdf"))
plt.show()